# Penyelarasan Data Temporal — Berita Geopolitik & Kurs USD/IDR

Notebook ini menyelaraskan data berita geopolitik (`geopolitical_news.csv`, hasil `00a_scraper.ipynb`) dengan data kurs harian USD/IDR dari Bank Indonesia (`jisdor_usd_idr.csv`, hasil `00c_timeseries_jisdor.ipynb`) berdasarkan waktu terbit beritanya.

## 0. Import Library

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_colwidth", 80)

## 1. Latar Belakang Masalah

Dua sumber data yang kita punya punya karakteristik waktu yang berbeda:

- **Data kurs (JISDOR)** hanya tersedia di hari kerja pasar sehingga tidak ada nilai di akhir pekan atau hari libur nasional.
- **Data berita** bisa terbit kapan saja, 24 jam, termasuk akhir pekan dan hari libur.

Akibatnya, sebuah berita yang terbit di hari Sabtu, Minggu, hari libur, atau setelah jam tutup pasar pada hari kerja, tidak punya "pasangan" langsung di data kurs pada tanggal yang sama persis. Notebook ini menetapkan aturan logis untuk memetakan tiap berita ke hari trading yang paling relevan, tanpa membuang informasinya.

## Load Data

In [ ]:
DATA_DIR = Path("../data")

import os
# "full" = articles_clean.csv (B_1) | "filtered" = articles_filtered.csv (B_2 dan C) | "id" = articles_id_clean.csv (B_3, CNBC Indonesia)
CORPUS = "en"
ART_FILE = {"full": "articles_clean.csv", "filtered": "articles_filtered.csv", "id": "articles_id_clean.csv"}[CORPUS]
OUT_FILE = {"full": "aligned_dataset.csv", "filtered": "aligned_dataset_filtered.csv", "id": "aligned_dataset_id.csv"}[CORPUS]

kurs = pd.read_csv(DATA_DIR / "processed" / "jisdor_usd_idr.csv", parse_dates=["date"])
kurs = kurs.sort_values("date").reset_index(drop=True)

news = pd.read_csv(DATA_DIR / "processed" / ART_FILE, parse_dates=["published_at"])  # sudah dibersihkan 00b_preprocessing.ipynb

print(f"Data kurs   : {len(kurs)} baris, {kurs['date'].min().date()} s.d. {kurs['date'].max().date()}")
print(f"Data berita : {len(news)} baris, {news['published_at'].min()} s.d. {news['published_at'].max()}")

kurs.head()

Data kurs   : 1202 baris, 2021-09-01 s.d. 2026-09-01
Data berita : 10075 baris, 2021-09-01 06:29:01 s.d. 2026-08-30 23:06:20


,date,usd_idr
0,2021-09-01,14284
1,2021-09-02,14281
2,2021-09-03,14261
3,2021-09-06,14239
4,2021-09-07,14195


In [3]:
news.head()

,published_at,date,clean_title,lead_paragraph,clean_content,dateline_location,word_count,sentence_count,paragraph_count,is_length_outlier,source_domain,language,url
0,2021-09-01 06:29:01,2021-09-01,Awas Panas! China Buat Aturan Sepihak di Laut China Selatan,"China mulai memberlakukan aturan barunya di Laut China Selatan hari ini, Rab...","China mulai memberlakukan aturan barunya di Laut China Selatan hari ini, Rab...",NaN,388,19,1,False,cnbcindonesia.com,id,https://www.cnbcindonesia.com/news/20210901062901-4-272738/awas-panas-china-...
1,2021-09-01 09:17:14,2021-09-01,"Selow, Inflasi Agustus Cuma 0,03%",Badan Pusat Statistik (BPS) mengumumkan data laju inflasi Indonesia periode ...,Badan Pusat Statistik (BPS) mengumumkan data laju inflasi Indonesia periode ...,NaN,142,10,1,False,cnbcindonesia.com,id,https://www.cnbcindonesia.com/news/20210901091714-4-272781/selow-inflasi-agu...
2,2021-09-01 11:13:40,2021-09-01,Bos BRI Sebut RI Siap Hadapi Risiko Tapering dari The Fed,"Kebijakan bank sentral Amerika Serikat, The Federal Reserve melakukan pengur...","Kebijakan bank sentral Amerika Serikat, The Federal Reserve melakukan pengur...",NaN,333,12,1,False,cnbcindonesia.com,id,https://www.cnbcindonesia.com/market/20210901111340-17-272837/bos-bri-sebut-...
3,2021-09-01 13:08:46,2021-09-01,BI: Kami Siap Antisipasi Dampak Tapering The Fed,Dampak pengurangan pembelian obligasi The Fed sebelum akhir tahun 2021 terus...,Dampak pengurangan pembelian obligasi The Fed sebelum akhir tahun 2021 terus...,NaN,101,5,1,False,cnbcindonesia.com,id,https://www.cnbcindonesia.com/market/20210901130846-19-272887/bi-kami-siap-a...
4,2021-09-01 15:12:18,2021-09-01,"Inflasi Benua Biru Melompat, Bursa Eropa Melaju di Zona Hijau","Bursa Eropa menguat pada pembukaan perdagangan Rabu (1/9/2021), meski rilis ...","Bursa Eropa menguat pada pembukaan perdagangan Rabu (1/9/2021), meski rilis ...",NaN,422,24,1,False,cnbcindonesia.com,id,https://www.cnbcindonesia.com/market/20210901151218-17-272936/inflasi-benua-...


## 3. Konversi Zona Waktu

Kolom `published_at` berasal dari `pubDate` RSS Google News, yang formatnya GMT (=UTC). Data kurs JISDOR mengikuti jam operasional pasar Indonesia (WIB, UTC+7). Supaya aturan "jam berapa market tutup" bisa diterapkan dengan benar, kedua data ini perlu disamakan zona waktunya dulu, kita konversi timestamp berita dari UTC ke WIB.

In [4]:
if CORPUS == "id":
    # waktu terbit CNBC Indonesia (dari URL) sudah WIB, jadi tidak boleh dikonversi dari UTC lagi
    news["published_wib"] = news["published_at"].dt.tz_localize("Asia/Jakarta")
else:
    news["published_utc"] = news["published_at"].dt.tz_localize("UTC")
    news["published_wib"] = news["published_utc"].dt.tz_convert("Asia/Jakarta")

news[["published_at", "published_wib"]].head()

,published_at,published_wib
0,2021-09-01 06:29:01,2021-09-01 06:29:01+07:00
1,2021-09-01 09:17:14,2021-09-01 09:17:14+07:00
2,2021-09-01 11:13:40,2021-09-01 11:13:40+07:00
3,2021-09-01 13:08:46,2021-09-01 13:08:46+07:00
4,2021-09-01 15:12:18,2021-09-01 15:12:18+07:00


## 4. Aturan Penyelarasan Temporal

Aturan yang digunakan (roll-forward ke hari trading berikutnya):

1. Daftar "hari trading valid" diambil langsung dari tanggal-tanggal yang ada di data kurs JISDOR, jadi hari libur nasional maupun akhir pekan otomatis terdeteksi tanpa perlu kalender libur terpisah, karena BI memang tidak mempublikasikan kurs pada hari-hari tersebut.
2. Ditetapkan jam cutoff 16:00 WIB sebagai asumsi jam tutup pasar valas domestik. Berita yang terbit sebelum jam ini pada hari kerja dianggap mempengaruhi kurs hari itu juga.
3. Berita yang terbit **setelah** jam cutoff, atau pada akhir pekan/hari libur, digeser maju ke hari trading valid berikutnya.
4. Berita tidak dihapus, hanya dipetakan ulang ke tanggal yang paling masuk akal secara logis, supaya sinyalnya tetap terpakai untuk pengujian hipotesis di tahap selanjutnya.

In [5]:
CUTOFF_JAM = 16  # asumsi: pasar valas domestik dianggap tutup jam 16:00 WIB

trading_days = pd.DatetimeIndex(kurs["date"].dt.normalize().unique())


def geser_ke_trading_day(ts, trading_days=trading_days, cutoff=CUTOFF_JAM):
    """Memetakan satu timestamp berita ke hari trading valid berikutnya."""
    tanggal = ts.normalize().tz_localize(None)
    if ts.hour >= cutoff:
        tanggal += pd.Timedelta(days=1)
    while tanggal not in trading_days:
        tanggal += pd.Timedelta(days=1)
        if tanggal > trading_days.max() + pd.Timedelta(days=10):
            return pd.NaT  # di luar cakupan data kurs
    return tanggal


news["trading_day"] = news["published_wib"].apply(geser_ke_trading_day)

n_gagal = news["trading_day"].isna().sum()
print(f"Total berita         : {len(news)}")
print(f"Berhasil dipetakan   : {len(news) - n_gagal}")
print(f"Gagal dipetakan      : {n_gagal} (di luar rentang tanggal data kurs)")

Total berita         : 10075
Berhasil dipetakan   : 10075
Gagal dipetakan      : 0 (di luar rentang tanggal data kurs)


## 5. Agregasi Berita per Hari Trading

Karena satu hari trading bisa menampung lebih dari satu berita (dan sebaliknya, satu hari trading bisa tidak punya berita sama sekali karena strategi *sampling* bi-weekly pada tahap scraping), berita perlu diagregasi dulu menjadi satu baris per hari sebelum digabung ke data kurs. Di sini kita hitung jumlah berita, menggabungkan judul-judulnya, dan menggabungkan isi teks bersihnya (clean_content) per hari sehingga hasil akhir bisa langsung dipakai untuk ekstraksi fitur teks di Tugas 2 tanpa perlu join ulang ke articles_clean.csv

In [6]:
def cap_daily_text(text_series, max_words=1500):
    combined = " | ".join(text_series.dropna())
    words = combined.split()
    return " | ".join(words[:max_words]) if len(words) > max_words else combined

agg = news.dropna(subset=["trading_day"]).groupby("trading_day").agg(
    jumlah_berita=("url", "count"),
    judul_gabungan=("clean_title", lambda x: " | ".join(x)),
    lead_paragraph_gabungan=("lead_paragraph", lambda x: cap_daily_text(x, max_words=1500)),
    konten_gabungan=("clean_content", lambda x: "\n\n---\n\n".join(x)),
).reset_index()

agg.head()

,trading_day,jumlah_berita,judul_gabungan,lead_paragraph_gabungan,konten_gabungan
0,2021-09-01,6,"Awas Panas! China Buat Aturan Sepihak di Laut China Selatan | Selow, Inflasi...","China mulai memberlakukan aturan barunya di Laut China Selatan hari ini, Rab...","China mulai memberlakukan aturan barunya di Laut China Selatan hari ini, Rab..."
1,2021-09-02,2,"Inflasi RI Masih Moderat, Yield Mayoritas SBN Kembali Turun | Covid Masih 'M...","Jakarta, CNBC Indonesia - Harga mayoritas obligasi pemerintah atau Surat Ber...","Jakarta, CNBC Indonesia - Harga mayoritas obligasi pemerintah atau Surat Ber..."
2,2021-09-06,2,"AS Diterjang Badai Ida, Joe Biden Terjun ke Lokasi! | Jreng.. Biden Minta Ka...","Joe Biden mengunjungi area terdampak Badai Ida di Louisiana, Amerika Serikat...","Joe Biden mengunjungi area terdampak Badai Ida di Louisiana, Amerika Serikat..."
3,2021-09-07,4,Mencekam! Begini Suasana Guinea Usai Militer Kudeta Presiden | Cadangan Devi...,Junta Militer menutup perbatasan di Guinea. Junta militer Guinea menutup per...,Junta Militer menutup perbatasan di Guinea. Junta militer Guinea menutup per...
4,2021-09-08,1,"Tiba-tiba Xi Jinping Bikin Bursa Saham di Beijing, buat Apa?",Presiden China Xi Jinping pada hari Kamis pekan lalu menyatakan negaranya ak...,Presiden China Xi Jinping pada hari Kamis pekan lalu menyatakan negaranya ak...


## 6. Penggabungan dengan Data Kurs 

In [7]:
final = kurs.merge(agg, left_on="date", right_on="trading_day", how="left")
final["jumlah_berita"] = final["jumlah_berita"].fillna(0).astype(int)
final["judul_gabungan"] = final["judul_gabungan"].fillna("")
final["lead_paragraph_gabungan"] = final["lead_paragraph_gabungan"].fillna("")
final["konten_gabungan"] = final["konten_gabungan"].fillna("")
final = final.drop(columns=["trading_day"])

final.head(10)

,date,usd_idr,jumlah_berita,judul_gabungan,lead_paragraph_gabungan,konten_gabungan
0,2021-09-01,14284,6,"Awas Panas! China Buat Aturan Sepihak di Laut China Selatan | Selow, Inflasi...","China mulai memberlakukan aturan barunya di Laut China Selatan hari ini, Rab...","China mulai memberlakukan aturan barunya di Laut China Selatan hari ini, Rab..."
1,2021-09-02,14281,2,"Inflasi RI Masih Moderat, Yield Mayoritas SBN Kembali Turun | Covid Masih 'M...","Jakarta, CNBC Indonesia - Harga mayoritas obligasi pemerintah atau Surat Ber...","Jakarta, CNBC Indonesia - Harga mayoritas obligasi pemerintah atau Surat Ber..."
2,2021-09-03,14261,0,,,
3,2021-09-06,14239,2,"AS Diterjang Badai Ida, Joe Biden Terjun ke Lokasi! | Jreng.. Biden Minta Ka...","Joe Biden mengunjungi area terdampak Badai Ida di Louisiana, Amerika Serikat...","Joe Biden mengunjungi area terdampak Badai Ida di Louisiana, Amerika Serikat..."
4,2021-09-07,14195,4,Mencekam! Begini Suasana Guinea Usai Militer Kudeta Presiden | Cadangan Devi...,Junta Militer menutup perbatasan di Guinea. Junta militer Guinea menutup per...,Junta Militer menutup perbatasan di Guinea. Junta militer Guinea menutup per...
5,2021-09-08,14266,1,"Tiba-tiba Xi Jinping Bikin Bursa Saham di Beijing, buat Apa?",Presiden China Xi Jinping pada hari Kamis pekan lalu menyatakan negaranya ak...,Presiden China Xi Jinping pada hari Kamis pekan lalu menyatakan negaranya ak...
6,2021-09-09,14272,0,,,
7,2021-09-10,14225,4,"Biden Ngamuk! Tiba-tiba Frustrasi Hilang Kesabaran, Ada Apa? | Geger Dunia! ...",Presiden Amerika Serikat (AS) Joe Biden tiba-tiba menunjukkan rasa ketidakse...,Presiden Amerika Serikat (AS) Joe Biden tiba-tiba menunjukkan rasa ketidakse...
8,2021-09-13,14260,12,"Stabilkan Harga, China Lelang Cadangan Minyak! | Ngeri! Korut Tembak Rudal 1...",China berencana melakukan lelang perdana cadangan minyak mentah kepada perus...,China berencana melakukan lelang perdana cadangan minyak mentah kepada perus...
9,2021-09-14,14257,2,"Ngeri...Ribuan Kapal China Masuki Natuna, Ganggu Tambang RI! | Ngeri! Potret...",Badan Keamanan Laut (Bakamla) menyebutkan bahwa ada ribuan kapal milik Vietn...,Badan Keamanan Laut (Bakamla) menyebutkan bahwa ada ribuan kapal milik Vietn...


## 7. Validasi & Ringkasan Hasil

In [8]:
total_hari = len(final)
hari_ada_berita = (final["jumlah_berita"] > 0).sum()

print(f"Total hari trading                : {total_hari}")
print(f"Hari trading dengan >=1 berita     : {hari_ada_berita} ({hari_ada_berita/total_hari:.1%})")
print(f"Hari trading tanpa berita          : {total_hari - hari_ada_berita}")
print(f"Total berita ter-assign            : {final['jumlah_berita'].sum()}")
print(f"Berita di luar cakupan data kurs   : {n_gagal} dari {len(news)} ({n_gagal/len(news):.1%})")
print()

Total hari trading                : 1202
Hari trading dengan >=1 berita     : 881 (73.3%)
Hari trading tanpa berita          : 321
Total berita ter-assign            : 10075
Berita di luar cakupan data kurs   : 0 dari 10075 (0.0%)



## 8. Menyimpan Hasil Akhir 

In [9]:
OUTPUT_PATH = DATA_DIR / "processed" / OUT_FILE
final.to_csv(OUTPUT_PATH, index=False)
print(f"Tersimpan di: {OUTPUT_PATH.resolve()}")
print(f"Bentuk akhir: {final.shape[0]} baris x {final.shape[1]} kolom")
final.dtypes

Tersimpan di: C:\UGM Stuff\Semester 5\Natural Language Processing\nlp-geopolitics-usa\data\processed\aligned_dataset_id.csv
Bentuk akhir: 1202 baris x 6 kolom


date                       datetime64[ns]
usd_idr                             int64
jumlah_berita                       int64
judul_gabungan                     object
lead_paragraph_gabungan            object
konten_gabungan                    object
dtype: object